<a href="https://colab.research.google.com/github/generativemodelingmva/generativemodelingmva.github.io/blob/main/tp2526/tp9_diffusion_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diffusion models: Sampling and conditional sampling of a pretrained DDPM model

This notebook is based on the [github repository](https://github.com/DPS2022/diffusion-posterior-sampling.git) of

*Diffusion Posterior Sampling for General Noisy Inverse Problems*, <br/>
by Hyungjin Chung, Jeongsol Kim, Michael T. Mccann, Marc L. Klasky, Jong Chul Ye, <br/>
ICLR 2023, https://arxiv.org/abs/2209.14687

We use the U-net trained by the authors on the FFHQ datasets (using the 49k images ```01000.png``` to ```49999.png```).

It is a DDPM model based on

* Diffusion Models Beat GANs on Image Synthesis, Prafulla Dhariwal, Alex Nichol, NeurIPS 2021, https://arxiv.org/abs/2105.05233
[github](https://github.com/openai/guided-diffusion/)

* Denoising Diffusion Probabilistic Models, Jonathan Ho, Ajay Jain, Pieter Abbeel, NeurIPS 2020, https://arxiv.org/abs/2006.11239
[projectpage](https://hojonathanho.github.io/diffusion/)

**Notebook authors:**
* Bruno Galerne: www.idpoisson.fr/galerne / https://github.com/bgalerne
* Arthur Leclaire: https://perso.telecom-paristech.fr/aleclaire/


$\newcommand{\bx}{\mathbf{x}} % bold x$
$\newcommand{\bz}{\mathbf{z}} % bold z$
$\newcommand{\bw}{\mathbf{w}} % bold w$


In [ ]:
# Download files
!git clone https://github.com/DPS2022/diffusion-posterior-sampling.git
!cp -r diffusion-posterior-sampling/guided_diffusion guided_diffusion
!wget -nc -O ffhq256-1k-validation.zip 'https://www.dropbox.com/scl/fi/pppstbdsf0em6o0qscruc/ffhq256-1k-validation.zip?rlkey=xl7nwv2nxb6yvsirr3wad77hm'
!unzip -nq ffhq256-1k-validation.zip
!wget -nc -O ffhq_10m.pt 'https://www.dropbox.com/scl/fi/pq72vxzxcbygieq5z4gvf/ffhq_10m.pt?rlkey=5sxdj6r4o9f7b7bbp5fxg2f5r'

In [ ]:
import torch
import torchvision
import numpy as np

import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm
from guided_diffusion.unet import create_model
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Display functions
We will work with PyTorch images with color values in $[-1,1]$ and the usual additional batch dimension, that is, images will have size 1x3x256x256 in PyTorch.

In [ ]:
def tensor2im(x):
    x = 0.5+0.5*x # [-1,1]->[0,1]
    return x.detach().cpu().permute(2,3,1,0).squeeze()

def im2tensor(x):
    x = torch.tensor(x,device=device)
    x = 2*x-1 # [0,1]->[-1,1]
    return x.permute(2,0,1).unsqueeze(0)

def rgb2gray(u):
    return 0.2989 * u[:,:,0] + 0.5870 * u[:,:,1] + 0.1140 * u[:,:,2]

def str2(chars):
    return "{:.2f}".format(chars)

def psnr(uref,ut,M=2):
    rmse = np.sqrt(np.mean((np.array(uref.cpu())-np.array(ut.cpu()))**2))
    return 20*np.log10(M/rmse)

# viewimage
import tempfile
import IPython

def viewimage(im, normalize=True,vmin=-1,vmax=1,z=1,order=0,titre='',displayfilename=False):
    im = im.detach().cpu().permute(2,3,1,0).squeeze()
    imin= np.array(im).astype(np.float32)
    channel_axis = 2 if len(im.shape)>2 else None
    if z!=1:
        from skimage.transform import rescale
        imin = rescale(imin, z, order=order, channel_axis=channel_axis)
    if normalize:
        if vmin is None:
            vmin = imin.min()
        if vmax is None:
            vmax = imin.max()
        if np.abs(vmax-vmin)>1e-10:
            imin = (imin.clip(vmin,vmax)-vmin)/(vmax-vmin)
        else:
            imin = vmin
    else:
        imin=imin.clip(0,255)/255
    imin=(imin*255).astype(np.uint8)
    filename=tempfile.mktemp(titre+'.png')
    if displayfilename:
        print (filename)
    plt.imsave(filename, imin, cmap='gray')
    IPython.display.display(IPython.display.Image(filename))

idx = np.random.randint(1000)
print('Image', str(idx).zfill(5))

# Test display function:
img = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'))
print(img.min().item(),img.max().item())
viewimage(img)

# Load DDPM Unet

In [ ]:
# Load model
model_config = {'image_size': 256,
                'num_channels': 128,
                'num_res_blocks': 1,
                'channel_mult': '',
                'learn_sigma': True,
                'class_cond': False,
                'use_checkpoint': False,
                'attention_resolutions': 16,
                'num_heads': 4,
                'num_head_channels': 64,
                'num_heads_upsample': -1,
                'use_scale_shift_norm': True,
                'dropout': 0.0,
                'resblock_updown': True,
                'use_fp16': False,
                'use_new_attention_order': False,
                'model_path': 'ffhq_10m.pt'}
model = create_model(**model_config)
model = model.to(device)
model.eval();  # use in eval mode

# DDPM parameters

The DDPM parameters defined in this cell will be used to sample the DDPM model.

The Unet ```model(x,t)``` estimates the residual noise $\varepsilon_t(\bx_t, t)$ from a noisy image $(\bx_t, t)$.

The function ```get_eps_from_model(x,t)``` computes $\varepsilon_t(\bx_t, t)$.

Remark that several variables are already precomputed in the class:
* `T` : $T$
* `reversed_time_steps` : backward $t$ range:  $\{T, T-1, \ldots, 1\}$
* `beta` : $(\beta_t)_{1 \leq t \leq T}$
* `alpha` : $(\alpha_t)_{1 \leq t \leq T}$
* `alphabar` : $(\bar{\alpha}_t)_{1 \leq t \leq T}$
* `alphabar_prev` : $(\bar{\alpha}_{t-1})_{1 \leq t \leq T}$


In [ ]:

T = 1000
time_steps = np.arange(T)
reversed_time_steps = time_steps[::-1]

beta_start = 0.0001
beta_end = 0.02
beta = np.linspace(beta_start, beta_end, T, dtype=np.float64)
alpha = 1.0 - beta
alphabar = np.cumprod(alpha)
betabar = 1-alphabar
alphabar_prev = np.append(1.0, alphabar[:-1])    # alpha_{t-1}

def get_eps_from_model(x, t):
    # the model outputs:
    # - an estimation of the noise eps (chanels 0 to 2)
    # - learnt variances for the posterior  (chanels 3 to 5)
    # (see Improved Denoising Diffusion Probabilistic Models
    # by Alex Nichol, Prafulla Dhariwal
    # for the parameterization)
    # We discard the second part of the output for this practice session.

    with torch.no_grad(): # avoid backprop wrt model parameters
        model_output = model(x, torch.tensor(t, device=device).unsqueeze(0))
    model_output = model_output[:,:3,:,:]
    return(model_output)


def get_eps_from_model_withgrad(x, t):
    # same as get_eps_from_model but kipping gradients
    model_output = model(x, torch.tensor(t, device=device).unsqueeze(0))
    model_output = model_output[:,:3,:,:]
    return(model_output)


<br/><br/><br/>

# Exercise 1: Forward Model

## Question 1: Sample the forward model

Take an image from FFHQ validation set and apply the forward model to it:
$$
\bx_{t+1} = \sqrt{\alpha_t} \bx_t + \sqrt{\beta_t} \bz_t
$$
Display $\bx_t$  for 10 different levels.

In [ ]:
#Exercise 1:
# Image 00462
idx = 462
img_pil = Image.open('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png')
x0 = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'))
imgshape = x0.shape

xt = x0.clone() # initialize xt for t=0

for t in range(T):

  with torch.no_grad(): # avoid backprop wrt model parameters

    # TODO: xt = ???


    if (t+1)%100==0 or t==0:
      print('Iteration :', t+1)
      viewimage(torch.cat((xt, x0), dim=3))


<br/><br/><br/>

## Question 2: Examine the prediction of $\bx_0$

Recall that we can obtain a prediction of $\bx_0$ from $\bx_t$:
$$
\widehat{\bx_0}(\bx_t) = \frac{1}{\sqrt{\bar{\alpha}_t}} \left( \bx_t - \sqrt{\bar{\beta}_t} \; \varepsilon_t(\bx_t, t) \right) ,
$$
which can also be written
$$
\widehat{\bx_0}(\bx_t) = \frac{1}{\sqrt{\bar{\alpha}_t}} \bx_t - \sqrt{\frac{1}{\bar{\alpha}_t}-1} \;  \varepsilon_t(\bx_t, t).
$$

Generate a sample $(\bx_t)$ of the forward model and the corresponding predictions $(\widehat{\bx_0}(\bx_t))$.

Display side by side $\bx_t$ and $\widehat{\bx_0}(\bx_t)$ every 100 iterations.

Display the two curves of the PSNR$(\bx_t,\bx_0)$ and PSNR$(\widehat{\bx_0}(\bx_t),\bx_0)$.


In [ ]:
psnrx = []
psnrxhat = []

x = x0.clone() # initialize xt for t=0

for t in range(T):

    # TODO

    if (t+1)%100==0 or t==0:
      print('Iteration :', t+1)
      viewimage(torch.cat((x, xhat, x0), dim=3))


# TODO Q2 plot


<br/><br/><br/>

# Exercise 2: Sample the backward model

Write a function `sample` that computes a sample of the backward model.

Let us recall that the DDPM transition probability is given by
$$ p_\theta(\bx_{t-1}|\bx_{t}) = \mathcal{N}(\mu_\theta(\bx_{t}, t), \beta_t I_d) $$
where
$$
\mu_\theta(\bx_t, t) = \frac{1}{\sqrt{\alpha_t}} \left(\bx_t - \frac{\beta_t}{\sqrt{\bar{\beta}_t}}\mathbf{\varepsilon}_\theta(\bx_t,t) \right).
$$

Display side by side $\bx_t$ and $\widehat{\bx_0}(\bx_t,t)$ every 100 iterations.

In [ ]:
x = torch.randn(imgshape,device=device)  # initialize x_t for t=T

for i, t in enumerate(reversed_time_steps):

    # TODO

    if i==0 or t%100==0 or t==0:
        print('Iteration:', i, '; Discrete time:', t)
        viewimage(torch.cat((x, xhat), dim=3))

<br/><br/><br/>

# Exercise 3: Conditional sampling for imaging inverse problems

In this exercise, we will solve an inverse problem with the "Diffusion Posterior Sampling" (DPS) algorithm proposed in

*Diffusion Posterior Sampling for General Noisy Inverse Problems*,<br/>
by Hyungjin Chung, Jeongsol Kim, Michael T. Mccann, Marc L. Klasky, Jong Chul Ye,<br/>
ICLR 2023, https://arxiv.org/abs/2209.14687

We restrict to linear measurements with Gaussian noise (eg inpainting, super-resolution, deblurring,...):
- $A$ is a linear operator
- $\eta$ is an additive Gaussian white noise
- the degraded observation (computed from a clean image $\bx$) is obtained as
$$\mathbf{y} = A \bx + \eta .$$

The DPS algorithm is designed to approximately sample the posterior distribution
$$
p_\theta(\bx_0| \mathbf{y} = A \bx_0 + \eta).
$$
This will be performed by adding a correction term to the DDPM backward sampling.



The cell below defines the forward operator and degraded observation for an inpainting experiment with a square mask.

In [ ]:
h = imgshape[2]
w = imgshape[3]
hcrop, wcrop = h//2, w//2
corner_top, corner_left = h//4, int(0.45*w)
mask = torch.ones(imgshape, device=device)
mask[:,:,corner_top:corner_top+hcrop,corner_left:corner_left+wcrop] = 0

def linear_operator(x):
  x = x*mask
  return(x)

x_true = x0.clone()
print("original image")
viewimage(x_true)

sigma_noise = 2*10/255

y = linear_operator(x_true.clone()) + sigma_noise * mask * torch.randn_like(x_true)
print("noisy measurement")
viewimage(y)


## Question 1

Implement the DPS algorithm with $\bx\mapsto A\bx$ implemented by `linear_operator(x)`:

> Initialize $x_T$ as for unconditional sampling.
>
> For $t=T$ to $1$:
>  1. Predict $\widehat{\bx_0}(\bx_t,t)$.
>  2. Compute the squared $\ell^2$ error $\|A\widehat{\bx_0}(\bx_t,t) - \mathbf{y} \|^2$ and its gradient w.r.t. $\bx_t$.
>  3. Define
>  $$ \bx_{t-1} = \mu_\theta(\bx_{t}, t) + \sqrt{\beta_t} \bz - \zeta_t \nabla_{\bx_t} \|A\widehat{\bx_0}(\bx_t,t) - \mathbf{y} \|^2. $$
> where the scaling factor $\zeta_t$ has been experimentally fixed as
$$
\zeta_t =
0.1\times \|A\widehat{\bx_0}(\bx_t,t) - \mathbf{y}\|^{-1}.
$$

Note that computing $\nabla_{\bx_t} \|A\widehat{\bx_0}(\bx_t,t) - \mathbf{y} \|^2$ involves a backpropagation through the Unet so one can expect the conditional sampling to be twice as long as the sampling procedure.

Display side by side $\bx_t, \widehat{\bx_0}(\bx_t,t), \mathbf{y}$ every 100 iterations, and maybe the groundtruth $x_0$. <br/>

In [ ]:
# visualization image for the observation y  (for inpainting, no resizing required)
vis_y = y

# initialize xt for t=T
x = torch.randn(imgshape, device=device,requires_grad=True)


for i, t in enumerate(reversed_time_steps):

  # TODO

  # Display
  if i==0 or t%100==0 or t==0:
      print('Iteration:', i, '; Discrete time:', t)
      viewimage(torch.cat((x, xhat, vis_y), dim=3))

## Question 2

Adapt the DPS algorithm in order to address a x4 super-resolution problem.

<br/><br/><br/>

# Exercise 4: ODE sampler: From DDPM to backward SDE to probability flow ODE

This exercise has been elaborated with the help of [Emile Pierret](https://www.idpoisson.fr/pierret) and is inspired by:
* *Score-Based Generative Modeling through Stochastic Differential Equations*,
Yang Song, Jascha Sohl-Dickstein, Diederik P. Kingma, Abhishek Kumar, Stefano Ermon, Ben Poole, ICLR 2021 https://arxiv.org/abs/2011.13456

As recalled in Exercise 2, the DDPM transition probability is given by
$$ p_\theta(\bx_{t-1}|\bx_{t}) = \mathcal{N}(\mu_\theta(\bx_{t}, t), \beta_t I_d) $$
where
$$
\mu_\theta(\bx_t, t) = \frac{1}{\sqrt{\alpha_t}} \left(\bx_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\mathbf{\varepsilon}_\theta(\bx_t,t) \right).
$$
This results in the sampling scheme:
$$
\bx_{t-1}
= \frac{1}{\sqrt{\alpha_t}} \left(\bx_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\mathbf{\varepsilon}_\theta(\bx_t,t) \right) + \sqrt{\beta_t} \bz_t,\quad \bz_t\sim \mathcal{N}(0, I_d).
$$
Now, by Tweedie formula we know that the score function is linked to the noise $\varepsilon$:
$$
\mathbb{E}\left[ \varepsilon|\bx_t\right] = - \sqrt{1-\bar{\alpha}_t} \nabla_{\bx_t} \log p_{t}(\bx_t).
$$
Hence we can defined the learnt score function as:
$$
s_\theta(\bx_t,t) = - \frac{1}{\sqrt{1-\bar{\alpha}_t}}\mathbf{\varepsilon}_\theta(\bx_t,t).
$$
The DDPM with the learnt score function is thus:
$$
\bx_{t-1}
= \frac{1}{\sqrt{\alpha_t}} \left(\bx_t + \beta_t s_\theta(\bx_t,t)\right) + \sqrt{\beta_t} \bz_t,\quad \bz_t\sim \mathcal{N}(0, I_d).
$$
Now since $\alpha_t = 1 -\beta_t$ with $\beta_t$ small by comparison of the time step $\Delta t=1$, we can approximate
$$
\frac{1}{\sqrt{\alpha_t}}
= \frac{1}{\sqrt{1 - \beta_t}}
\approx 1 + \frac{1}{2} \beta_t.
$$
This gives the approximate sampling scheme:
$$
\bx_{t-1}
= \bx_t + \frac{1}{2} \beta_t x_t + \beta_t s_\theta(\bx_t,t) + \sqrt{\beta_t} \bz_t,\quad \bz_t\sim \mathcal{N}(0, I_d).
$$
This is an Euler-Maruyama scheme with step $\Delta t=1$ for the diffusion SDE:
$$
d\bx_t = \frac{1}{2} \beta_t x_t + \beta_t \nabla_\bx\log  p_t(\bx_t) + \sqrt{\beta_t} d \bw_t
$$
with the learnt score function $s_\theta$.
The corresponding probability flow ODE is:
$$
d\bx_t = \frac{1}{2} \beta_t x_t + \frac{1}{2}  \beta_t \nabla_\bx\log  p_t(\bx_t).
$$
**Questions:**

1. Define a function ```ode_euler_sampling``` with prototype <br/>
`ode_euler_sampling(nb_times_integration=1000, noise_seed = None, show_steps=True)` <br/>
that samples the above probability flow ODE with ```nb_times_integration```steps (supposed to be a divisor of $T=1000$) and accepts a seed to draw the initial Gaussian noise at $T=1000$.  <br/>
Test the function with various seeds.
2. Test the ```ode_euler_sampling``` function with the same seed with the various number of steps 20, 50, 100 and 200, 1000. Comment the results.
3. Define a function ```ode_euler_samples_interpolation``` with prototype <br/>
`ode_euler_samples_interpolation(nb_times_integration=100, noise_seed0 = None, noise_seed1 = None, n_interm = 7)` <br/>
that performs samples interpolation by linearly interpolating the initial noises:
$$
\bx_T^\gamma =  (1-\gamma) \bx_T^0 + \gamma \bx_T^1.
$$
Are the interpolation results satisfying ?
4. Propose and test a better solution for sample interpolation.


In [ ]:
# Question 4.1)
# TODO


In [ ]:
ode_euler_sampling(nb_times_integration=100,
                        noise_seed = 20250230,
                        show_steps=True);

In [ ]:
# Question 4.2)
# TODO



In [ ]:
# Question 4.3)
# TODO



In [ ]:

samples_list = ode_euler_samples_interpolation(nb_times_integration=100,
                                                    noise_seed0 = 20250225,
                                                    noise_seed1 = 20250230,
                                                    n_interm = 7);
viewimage(torch.cat(samples_list, dim=3))